<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Sıfırdan Byte Pair Encoding (BPE) Tokenizer — Basit

- Bu, GPT-2'den GPT-4'e, Llama 3 gibi modellerde kullanılan popüler byte pair encoding (BPE) token'lara ayırma algoritmasını eğitim amacıyla sıfırdan uygulayan bağımsız bir not defteridir
- Token'lara ayırmanın amacı hakkında daha fazla ayrıntı için lütfen [Bölüm 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb) dosyasına bakın; buradaki kod BPE algoritmasını açıklayan bonus materyaldir
- OpenAI'ın orijinal GPT modellerini eğitmek için uyguladığı özgün BPE tokenizer'ına [buradan](https://github.com/openai/gpt-2/blob/master/src/encoder.py) ulaşabilirsiniz
- BPE algoritması ilk kez 1994'te tanımlanmıştır: Philip Gage, "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- Llama 3 dahil çoğu proje bugünlerde hesaplama başarımı nedeniyle OpenAI'ın açık kaynaklı [tiktoken kütüphanesini](https://github.com/openai/tiktoken) kullanıyor; bu kütüphane örneğin önceden eğitilmiş GPT-2 ve GPT-4 tokenizer'larını yüklemeye olanak tanır (Llama 3 modelleri de GPT-4 tokenizer'ı kullanılarak eğitilmiştir)
- Yukarıdaki uygulamalarla bu not defterindeki uygulamam arasındaki fark, buradakinin (eğitim amacıyla) tokenizer'ı eğitmek için bir fonksiyon da içermesidir
- Eğitim desteği olan ve muhtemelen daha başarımlı olan [minBPE](https://github.com/karpathy/minbpe) adlı bir uygulama da var (buradaki uygulamam eğitim amacına odaklıdır); `minbpe`'den farklı olarak benim uygulamam ayrıca orijinal OpenAI tokenizer sözlüğünü ve birleştirmelerini (merges) yüklemeye de izin verir

**Bu, eğitim amaçlı çok yalın bir uygulamadır. [bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) not defteri, tiktoken'ın davranışıyla örtüşen daha gelişmiş (ama okunması çok daha zor) bir uygulama içerir.**

&nbsp;
# 1. Byte pair encoding'in (BPE) arkasındaki temel fikir

- BPE'deki temel fikir, LLM eğitimi için metni bir tam sayı temsiline (token kimlikleri) dönüştürmektir (bkz. [Bölüm 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

&nbsp;
## 1.1 Bit'ler ve bayt'lar

- BPE algoritmasına geçmeden önce bayt (byte) kavramını tanıtalım
- Metni bir bayt dizisine dönüştürmeyi düşünün (nihayetinde BPE, "byte" pair encoding'in kısaltması):

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- Bir `bytearray` nesnesi üzerinde `list()` çağırdığımızda her bayt ayrı bir eleman olarak ele alınır ve sonuç, bayt değerlerine karşılık gelen bir tam sayı listesi olur:

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- Bu, metni bir LLM'in gömme katmanı için ihtiyaç duyduğumuz token kimliği temsiline dönüştürmenin geçerli bir yolu olurdu
- Ancak bu yaklaşımın dezavantajı, her karakter için bir kimlik oluşturmasıdır (kısa bir metin için bile bu çok fazla kimlik demek!)
- Yani 17 karakterlik bir girdi metni için LLM'e girdi olarak 17 token kimliği kullanmamız gerekir:

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- Daha önce LLM'lerle çalıştıysanız, BPE tokenizer'larının her karakter yerine tam kelimeler veya alt kelimeler için token kimliği içeren bir sözlüğe sahip olduğunu biliyor olabilirsiniz
- Örneğin GPT-2 tokenizer'ı aynı metni ("This is some text") 17 yerine yalnızca 4 token'a ayırır: `1212, 318, 617, 2420`
- Bunu etkileşimli [tiktoken uygulamasını](https://tiktokenizer.vercel.app/?model=gpt2) veya [tiktoken kütüphanesini](https://github.com/openai/tiktoken) kullanarak doğrulayabilirsiniz:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

- Bir bayt 8 bit'ten oluştuğu için, tek bir baytın temsil edebileceği 2<sup>8</sup> = 256 olası değer vardır; bunlar 0 ile 255 arasındadır
- Bunu `bytearray(range(0, 257))` kodunu çalıştırarak doğrulayabilirsiniz; bu kod size `ValueError: byte must be in range(0, 256)` uyarısını verecektir
- Bir BPE tokenizer'ı genellikle bu 256 değeri ilk 256 tek karakterli token'ı olarak kullanır; bunu şu kodu çalıştırarak gözle kontrol edebilirsiniz:

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- single character tokens up to here
256:  t
257:  a
...
298: ent
299:  n
"""
```

- Yukarıda, 256 ve 257 numaralı kayıtların tek karakterli değil çift karakterli değerler olduğuna dikkat edin (bir boşluk + bir harf); bu, orijinal GPT-2 BPE tokenizer'ının küçük bir eksikliğidir (GPT-4 tokenizer'ında düzeltilmiştir)

&nbsp;
## 1.2 Sözlüğü oluşturmak

- BPE token'lara ayırma algoritmasının amacı, `298: ent` gibi sık geçen alt kelimelerden (örneğin *entangle, entertain, enter, entrance, entity, ...* içinde bulunabilir) ve hatta tam kelimelerden oluşan bir sözlük oluşturmaktır:

```
318: is
617: some
1212: This
2420: text
```

- BPE algoritması ilk kez 1994'te tanımlanmıştır: Philip Gage, "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- Asıl kod uygulamasına geçmeden önce, bugün LLM tokenizer'larında kullanılan biçim şöyle özetlenebilir:

&nbsp;
## 1.3 BPE algoritmasının ana hatları

**1. Sık geçen çiftleri belirle**
- Her yinelemede metni tarayarak en sık geçen bayt (veya karakter) çiftini bul

**2. Değiştir ve kaydet**

- Bu çifti, henüz kullanılmayan yeni bir yer tutucu kimlikle değiştir (ör. 0...255 ile başlıyorsak ilk yer tutucu 256 olur)
- Bu eşlemeyi bir arama tablosuna kaydet
- Arama tablosunun boyutu bir hiperparametredir; buna "sözlük boyutu" (vocabulary size) da denir (GPT-2 için bu 50.257'dir)

**3. Kazanç kalmayana kadar tekrarla**

- 1. ve 2. adımları, en sık geçen çiftleri sürekli birleştirerek tekrarla
- Daha fazla sıkıştırma mümkün olmadığında dur (ör. hiçbir çift birden fazla geçmiyorsa)

**Açma (kod çözme)**

- Orijinal metni geri getirmek için, arama tablosunu kullanarak her kimliği karşılık gelen çiftle değiştirip süreci tersine çevir



&nbsp;
## 1.4 BPE algoritması örneği

### 1.4.1 Kodlama kısmının somut örneği (1. ve 2. adımlar)

- Elimizde, bir BPE tokenizer'ı için sözlük oluşturmak istediğimiz `the cat in the hat` metni (eğitim veri kümesi) olduğunu varsayalım

**1. Yineleme**

1. Sık geçen çiftleri belirle
  - Bu metinde "th" iki kez geçiyor (başta ve ikinci "e"den önce)

2. Değiştir ve kaydet
  - "th" ifadesini henüz kullanılmayan yeni bir token kimliğiyle, ör. 256 ile değiştir
  - yeni metin: `<256>e cat in <256>e hat`
  - yeni sözlük:

```
  0: ...
  ...
  256: "th"
```

**2. Yineleme**

1. **Sık geçen çiftleri belirle**  
   - `<256>e cat in <256>e hat` metninde `<256>e` çifti iki kez geçiyor

2. **Değiştir ve kaydet**  
   - `<256>e` ifadesini henüz kullanılmayan yeni bir token kimliğiyle, örneğin `257` ile değiştir.  
   - Yeni metin:
     ```
     <257> cat in <257> hat
     ```
   - Güncellenen sözlük:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**3. Yineleme**

1. **Sık geçen çiftleri belirle**  
   - `<257> cat in <257> hat` metninde `<257> ` çifti iki kez geçiyor (biri başta, biri "hat" öncesinde).

2. **Değiştir ve kaydet**  
   - `<257> ` ifadesini henüz kullanılmayan yeni bir token kimliğiyle, örneğin `258` ile değiştir.  
   - yeni metin:
     ```
     <258>cat in <258>hat
     ```
   - Güncellenen sözlük:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- ve bu böyle devam eder

&nbsp;
### 1.4.2 Kod çözme kısmının somut örneği (3. adım)

- Orijinal metni geri getirmek için, her token kimliğini karşılık gelen çiftle, eklendikleri sıranın tersinden başlayarak değiştirip süreci tersine çeviririz
- Son sıkıştırılmış metinle başla: `<258>cat in <258>hat`
-  `<258>` → `<257> ` yerine koy: `<257> cat in <257> hat`  
- `<257>` → `<256>e` yerine koy: `<256>e cat in <256>e hat`
- `<256>` → "th" yerine koy: `the cat in the hat`

&nbsp;
## 2. Basit bir BPE uygulaması

- Aşağıda, yukarıda anlatılan bu algoritmanın `tiktoken` Python arayüzünü taklit eden bir Python sınıfı olarak uygulaması yer alıyor
- Yukarıdaki kodlama kısmının `train()` üzerinden orijinal eğitim adımını anlattığını unutmayın; ancak `encode()` metodu da benzer şekilde çalışır (özel token'ların ele alınması nedeniyle biraz daha karmaşık görünse de):

1. Girdi metnini tek tek baytlara böl
2. Öğrenilen BPE birleştirmelerinden herhangi biriyle eşleşen komşu token'ları (çiftleri) tekrar tekrar bul ve değiştir (birleştir) — en yüksek "sıradan" (rank) en düşüğe doğru, yani öğrenilme sırasına göre
3. Uygulanabilecek birleştirme kalmayana kadar birleştirmeye devam et
4. Elde edilen nihai token kimliği listesi kodlanmış çıktıdır

In [ ]:
from collections import Counter, deque
from functools import lru_cache


class BPETokenizerSimple:
    def __init__(self):
        # token_id -> token_str eşlemesi (ör. {11246: "some"})
        self.vocab = {}
        # token_str -> token_id eşlemesi (ör. {"some": 11246})
        self.inverse_vocab = {}
        # BPE birleştirmeleri sözlüğü: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Ön işleme: Boşlukları 'Ġ' ile değiştir
        # Ġ karakterinin GPT-2 BPE uygulamasına özgü olduğunu unutmayın
        # Ör. "Hello world" şu şekilde token'lara ayrılabilir: ["Hello", "Ġworld"]
        # (GPT-4 BPE bunu ["Hello", " world"] olarak ayırırdı)
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Sözlüğü benzersiz karakterlerle başlat, varsa 'Ġ' dahil
        # İlk 256 ASCII karakteriyle başla
        unique_chars = [chr(i) for i in range(256)]

        # unique_chars listesini processed_text içindeki henüz eklenmemiş karakterlerle genişlet
        unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

        # İsteğe bağlı: metin işleme akışın için gerekliyse 'Ġ' karakterinin dahil olduğundan emin ol
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')

        # Şimdi sözlük ve ters sözlük (inverse vocab) sözlüklerini oluştur
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # İzin verilen özel token'ları ekle
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # processed_text metnini token kimliklerine ayır
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE 1-3. adımlar: Sık geçen çiftleri tekrar tekrar bul ve değiştir
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:  # No more pairs to merge. Stopping training.
                break
            
            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id

        # Sözlüğü birleştirilmiş token'larla oluştur
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text):
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # Metni token'lara böl, satır sonlarını olduğu gibi koru
        words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word)  # Handle first word or standalone '\n'

        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token sözlükte olduğu gibi mevcut
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # BPE ile alt kelime token'lara ayırmayı dene
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Token'ı tek tek karakterlere ayır (başlangıç token kimlikleri olarak)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # Eğitim amaçlı olarak yorumu kaldırabilirsiniz:
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # 'Ġ' karakterini boşlukla değiştir
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        if(len(token_ids) < 2):
            return None
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Çiftin 2. token'ını kaldır, 1. zaten kaldırılmıştı
                dq.popleft()
            else:
                replaced.append(current)

        return replaced


### Kısa token dizileri için uç durum yönetimi

BPE birleştirmeleri komşu token çiftleri gerektirir.  
Token dizisinde 2'den az öğe varsa çift oluşmaz; bu durumda `find_freq_pair` fonksiyonu `None` döndürür ve eğitim sorunsuz biçimde durur.

In [21]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([42]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


- Yukarıdaki `BPETokenizerSimple` sınıfında epey kod var ve bunu ayrıntılı tartışmak bu not defterinin kapsamı dışında; ancak bir sonraki bölüm, sınıf metotlarını biraz daha iyi anlamak için kullanıma dair kısa bir genel bakış sunuyor

## 3. BPE uygulamasının adım adım incelenmesi

- Pratikte [tiktoken](https://github.com/openai/tiktoken) kullanmanızı şiddetle öneririm; çünkü yukarıdaki uygulamam başarıma değil, okunabilirliğe ve eğitim amacına odaklıdır
- Yine de kullanımı aşağı yukarı tiktoken'a benzer; tek fark tiktoken'ın bir eğitim metodu olmamasıdır
- Yukarıdaki `BPETokenizerSimple` Python kodumun nasıl çalıştığını aşağıdaki örneklere bakarak görelim (ayrıntılı kod tartışması bu not defterinin kapsamı dışındadır)

### 3.1 Eğitim, kodlama ve kod çözme

- Önce eğitim veri kümemiz olarak bir örnek metin ele alalım:

In [5]:
import os
import urllib.request

if not os.path.exists("../01_main-chapter-code/the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "../01_main-chapter-code/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

- Ardından BPE tokenizer'ını 1.000 sözlük boyutuyla başlatıp eğitelim
- Daha önce ele alınan bayt değerleri nedeniyle sözlük boyutunun varsayılan olarak zaten 255 olduğunu unutmayın; yani yalnızca 745 sözlük kaydını "öğreniyoruz"
- Karşılaştırma için: GPT-2 sözlüğü 50.257 token, GPT-4 sözlüğü 100.256 token (tiktoken'da `cl100k_base`), GPT-4o ise 199.997 token (tiktoken'da `o200k_base`) içerir; hepsinin eğitim kümeleri yukarıdaki basit örnek metnimize kıyasla çok daha büyüktür

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- Sözlük içeriğini incelemek isteyebilirsiniz (ancak bunun uzun bir liste oluşturacağını unutmayın)

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- Bu sözlük 742 kez birleştirme yapılarak oluşturulmuştur (~ `1000 - len(range(0, 256))`)

In [8]:
print(len(tokenizer.bpe_merges))

742


- Bu, ilk 256 kaydın tek karakterli token'lar olduğu anlamına gelir

- Şimdi, oluşturulan birleştirmeleri `encode` metodu üzerinden kullanarak bir metni kodlayalım:

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 42
Number of token IDs: 20


- Yukarıdaki uzunluklardan görebiliyoruz ki 42 karakterlik bir cümle 20 token kimliğine kodlandı; bu, karakter-bayt tabanlı bir kodlamaya kıyasla girdi uzunluğunu kabaca yarıya indiriyor

- Sözlüğün kendisinin `decode()` metodunda kullanıldığını ve token kimliklerini tekrar metne eşlememizi sağladığını unutmayın:

In [11]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [12]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


- Her token kimliği üzerinde tek tek dolaşmak, token kimliklerinin sözlük aracılığıyla nasıl çözüldüğünü daha iyi anlamamızı sağlar:

In [13]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .


- Görüldüğü gibi token kimliklerinin çoğu 2 karakterli alt kelimeleri temsil ediyor; bunun nedeni eğitim verisi metninin çok kısa olması, çok fazla tekrar eden kelime içermemesi ve nispeten küçük bir sözlük boyutu kullanmamızdır

- Özetle, `decode(encode())` çağrısı herhangi bir girdi metnini yeniden üretebilmelidir:

In [14]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

&nbsp;
# 4. Sonuç

- İşte bu kadar! BPE özetle böyle çalışır; üstelik yeni tokenizer'lar oluşturmak için bir eğitim metoduyla birlikte
- Umarım bu kısa öğreticiyi eğitim amacıyla faydalı bulmuşsunuzdur; sorularınız varsa [buradan](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a) yeni bir Discussion açmaktan çekinmeyin


**Bu, eğitim amaçlı çok yalın bir uygulamadır. [bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) not defteri, tiktoken'ın davranışıyla örtüşen daha gelişmiş (ama okunması çok daha zor) bir uygulama içerir.**